# Predictive Modeling: Feature Engineering

**Goal:** build one zip-month feature set for predicting food assistance (CalFresh dollars and food bank pounds) from LIHEAP enrollment, census demographics, time and policy context.

**Inputs** (all in the repository root):
- `TimeLag_PolicyAnnotated.csv`: LIHEAP enrollment with lags 1–3 and the policy annotations from Assignment 4
- `CalFresh_cleaned.csv`, `FoodBank_cleaned.csv`: outcomes, plus their history for year-over-year features
- `Demographics_SD_ZipCode.csv`: ACS 5-year census estimates by zip code

**Outputs:**
- `Predictive_Features.csv`: one row per zip-month with the targets and all features (missing values handled by the rules in Section 8)
- `Feature_Dictionary.csv`: each feature's group, rationale and missing-value rule

**Feature groups**
1. LIHEAP lag features (t-1, t-2, t-3)
2. Demographic indicators from census data
3. Temporal features (month, quarter, year-over-year change)
4. Policy change binary indicators (Assignment 4)
5. Interaction terms

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", "{:.3f}".format)
pd.set_option("display.max_columns", 60)

KEY = ["Zip Code", "Month_Year"]
LIHEAP_WINDOW = ("2023-01", "2023-07")   # months with usable zip-level LIHEAP data
DEMO_MAX_YEAR = 2023                     # latest ACS vintage allowed (no look-ahead)

## 1. Load data

In [2]:
base = pd.read_csv("TimeLag_PolicyAnnotated.csv")
calfresh = pd.read_csv("CalFresh_cleaned.csv")
foodbank = pd.read_csv("FoodBank_cleaned.csv")
demo_raw = pd.read_csv("Demographics_SD_ZipCode.csv")

base["Zip Code"] = base["Zip Code"].astype(int)
base["Period"] = pd.PeriodIndex(base["Month_Year"], freq="M")

print(f"Base table: {base.shape[0]} zip-months, {base['Zip Code'].nunique()} zips, "
      f"{base['Month_Year'].min()} to {base['Month_Year'].max()}")

Base table: 852 zip-months, 121 zips, 2023-02 to 2023-10


## 2. Targets and outcome history

CalFresh dollars and food bank pounds are summed to one value per zip-month. Food bank months with zero pounds are treated as missing (no distribution recorded), matching `Correlation_Analysis.ipynb`. The full history (CalFresh from 2020, food bank from 2018) is kept so year-over-year features can look back 12–13 months.

In [3]:
cf_hist = (calfresh.groupby(["Zip Code", "Month Year"], as_index=False)["Calfresh Dollars"].sum()
           .rename(columns={"Month Year": "Month_Year", "Calfresh Dollars": "CalFresh_Dollars"}))
fb_hist = (foodbank.groupby(["Zip Code", "Month Year"], as_index=False)["Pounds_Distributed"].sum()
           .rename(columns={"Month Year": "Month_Year", "Pounds_Distributed": "FoodBank_Pounds"}))
fb_hist.loc[fb_hist["FoodBank_Pounds"] == 0, "FoodBank_Pounds"] = np.nan

for h in (cf_hist, fb_hist):
    h["Zip Code"] = h["Zip Code"].astype(int)
    h["Period"] = pd.PeriodIndex(h["Month_Year"], freq="M")

df = (base.merge(cf_hist[["Zip Code", "Period", "CalFresh_Dollars"]], on=["Zip Code", "Period"], how="left")
          .merge(fb_hist[["Zip Code", "Period", "FoodBank_Pounds"]], on=["Zip Code", "Period"], how="left"))
assert len(df) == len(base)

TARGETS = ["CalFresh_Dollars", "FoodBank_Pounds"]
df[TARGETS].describe().T

,count,mean,std,min,25%,50%,75%,max
CalFresh_Dollars,850.000,750573.941,841648.763,3470.000,151896.365,489653.000,920151.250,4724991.000
FoodBank_Pounds,681.000,82382.304,111292.857,424.000,16186.000,47236.391,104791.702,776447.031


## 3. LIHEAP lag features (t-1, t-2, t-3)

**Rationale:** LIHEAP enrollment signals household energy hardship, which can precede food assistance demand by one to three months. The correlation analysis found the same-month and one-month lags most informative within zip codes, so all three lags are kept for the model to weigh.

**Missing values.** The `lagk_status` columns from `TimeLag_Generation.ipynb` say *why* a lag is empty, and the two cases are handled differently:
- `zip_gap`: the month is inside the LIHEAP data window but the zip had no enrollment record, so it is a true zero → set to **0**.
- `before_start` / `after_end`: the month falls outside the window (Jan–Jul 2023), so the value is unknown → left **missing**, with a `_observed` flag (1 = value known).

Two summary features are added: the 3-month mean of available lags (smooths noisy monthly counts) and the lag-1 minus lag-2 change (short-term momentum).

In [4]:
LAGS = {1: "LIHEAP_lag1", 2: "LIHEAP_lag2", 3: "LIHEAP_lag3"}

for k, col in LAGS.items():
    status = df[f"lag{k}_status"]
    df.loc[status == "zip_gap", col] = 0.0                      # structural zero
    df.loc[status.isin(["before_start", "after_end"]), col] = np.nan
    df[f"{col}_observed"] = status.isin(["present", "zip_gap"]).astype(int)

# Same-month enrollment: zero if inside the window but unrecorded, missing outside it
in_window = df["Month_Year"].between(*LIHEAP_WINDOW)
df["LIHEAP_t0"] = np.where(in_window, df["Enrollment_Count"].fillna(0), np.nan)
df["LIHEAP_t0_observed"] = in_window.astype(int)

df["LIHEAP_lag_mean_3"] = df[list(LAGS.values())].mean(axis=1, skipna=True)
df["LIHEAP_lag_momentum"] = df["LIHEAP_lag1"] - df["LIHEAP_lag2"]

LAG_FEATURES = (["LIHEAP_t0"] + list(LAGS.values()) + ["LIHEAP_lag_mean_3", "LIHEAP_lag_momentum"]
                + ["LIHEAP_t0_observed"] + [f"{c}_observed" for c in LAGS.values()])
df[LAG_FEATURES].describe().T

,count,mean,std,min,25%,50%,75%,max
LIHEAP_t0,589.000,7.509,10.002,0.000,1.000,5.000,11.000,75.000
LIHEAP_lag1,687.000,7.247,9.527,0.000,1.000,5.000,10.000,75.000
LIHEAP_lag2,698.000,7.133,9.495,0.000,1.000,4.000,10.000,75.000
LIHEAP_lag3,679.000,7.333,9.551,0.000,1.000,5.000,10.000,75.000
LIHEAP_lag_mean_3,852.000,7.003,8.535,0.333,1.000,5.000,9.333,72.333
LIHEAP_lag_momentum,609.000,-0.082,5.440,-36.000,-2.000,0.000,2.000,32.000
LIHEAP_t0_observed,852.000,0.691,0.462,0.000,0.000,1.000,1.000,1.000
LIHEAP_lag1_observed,852.000,0.806,0.395,0.000,1.000,1.000,1.000,1.000
LIHEAP_lag2_observed,852.000,0.819,0.385,0.000,1.000,1.000,1.000,1.000
LIHEAP_lag3_observed,852.000,0.797,0.403,0.000,1.000,1.000,1.000,1.000


## 4. Demographic indicators (census)

**Rationale:** need for both energy and food assistance tracks poverty, unemployment, housing costs and household makeup. The demographic analysis also found that the LIHEAP–food bank relationship depends strongly on poverty.

- One ACS vintage per zip code: the latest year up to 2023 with complete subject-table data, so no future information is used.
- Counts are converted to **rates** (shares of population or households). Raw counts would mostly measure zip size, which the correlation analysis showed inflates relationships.
- `Log_Households` is kept as an explicit size control, and LIHEAP is also expressed per 1,000 households.

**Missing values:** zips with no ACS match (mostly PO-box zips) get the median across zip codes for each indicator, plus a `Demo_Imputed` flag so the model can tell imputed rows apart.

In [5]:
demo = (demo_raw[(demo_raw["Data_Year"] <= DEMO_MAX_YEAR)
                 & (~demo_raw["Has_Missing_Demographic_Data"].astype(bool))]
        .sort_values("Data_Year")
        .groupby("Zip Code", as_index=False).tail(1)
        .copy())
demo["Zip Code"] = demo["Zip Code"].astype(int)

pop = demo["Total_Population"].replace(0, np.nan)
hh = demo["Total_Households"].replace(0, np.nan)
demo = demo.assign(
    Poverty_Rate=demo["Poverty_Rate_Pct"],
    Unemployment_Rate=demo["Unemployment_Rate_Pct"],
    Median_Rent_k=demo["Median_Gross_Rent"] / 1000,
    Pct_Under_18=demo["Population_Under_18"] / pop * 100,
    Pct_65_Plus=demo["Population_65_Plus"] / pop * 100,
    Pct_Single_Parent_HH=(demo["Single_Father_Households_With_Own_Children_Under18"]
                          + demo["Single_Mother_Households_With_Own_Children_Under18"]) / hh * 100,
    Pct_Living_Alone=demo["Householder_Living_Alone"] / hh * 100,
    Avg_HH_Size=pop / hh,
    Log_Households=np.log1p(demo["Total_Households"]),
)

DEMO_FEATURES = ["Poverty_Rate", "Unemployment_Rate", "Median_Rent_k", "Pct_Under_18", "Pct_65_Plus",
                 "Pct_Single_Parent_HH", "Pct_Living_Alone", "Avg_HH_Size", "Log_Households"]

df = df.merge(demo[["Zip Code", "Total_Households"] + DEMO_FEATURES], on="Zip Code",
              how="left", validate="many_to_one")

df["Demo_Imputed"] = df["Poverty_Rate"].isna().astype(int)
zip_medians = demo[DEMO_FEATURES].median()
df[DEMO_FEATURES] = df[DEMO_FEATURES].fillna(zip_medians)

# LIHEAP normalized by zip size (households); missing when household count is unknown
df["LIHEAP_lag1_per_1k_HH"] = df["LIHEAP_lag1"] / df["Total_Households"] * 1000

print(f"Zips without census data (median-imputed): "
      f"{df.loc[df['Demo_Imputed'] == 1, 'Zip Code'].nunique()} of {df['Zip Code'].nunique()}")
df[DEMO_FEATURES].describe().T

Zips without census data (median-imputed): 32 of 121


,count,mean,std,min,25%,50%,75%,max
Poverty_Rate,852.000,10.153,4.307,2.900,7.400,9.600,11.800,21.800
Unemployment_Rate,852.000,5.815,1.890,0.000,4.700,5.400,6.600,12.400
Median_Rent_k,852.000,2.208,0.457,0.986,1.912,2.143,2.462,3.501
Pct_Under_18,852.000,21.215,4.823,4.190,19.074,21.955,23.902,33.128
Pct_65_Plus,852.000,16.180,5.114,6.676,13.447,16.173,17.863,42.531
Pct_Single_Parent_HH,852.000,7.022,2.789,1.275,5.510,6.340,8.634,15.786
Pct_Living_Alone,852.000,23.265,8.346,9.972,18.329,22.028,25.694,59.215
Avg_HH_Size,852.000,2.827,0.404,1.680,2.643,2.822,3.070,3.613
Log_Households,852.000,9.316,0.867,5.707,9.259,9.482,9.811,10.264


## 5. Temporal features (month, quarter, year-over-year change)

**Rationale:** food assistance has seasonal patterns (for example holiday-season food bank demand), and 2023 included program-wide shifts.
- `Month`, `Quarter`: calendar seasonality. `Month_sin` and `Month_cos` encode month as a cycle, so December and January sit next to each other.
- `Months_Since_Start`: linear trend across the study window.
- **Year-over-year change** uses only *past* outcome values, so the target never leaks into its own features:
  - `*_lag1`: last month's value (autoregressive baseline)
  - `*_same_month_last_year`: value 12 months earlier (seasonal baseline)
  - `*_yoy_change_lag1`: last month vs. the same month a year before, (t-1 ÷ t-13) − 1

**Missing values:** history features are missing when the zip had no record at that earlier month. They are left missing here; Section 8 fills them with 0 and adds a flag.

In [6]:
df["Month"] = df["Period"].dt.month
df["Quarter"] = df["Period"].dt.quarter
df["Month_sin"] = np.sin(2 * np.pi * df["Month"] / 12)
df["Month_cos"] = np.cos(2 * np.pi * df["Month"] / 12)
df["Months_Since_Start"] = (df["Period"] - df["Period"].min()).apply(lambda x: x.n)


def lagged_value(df, hist, value_col, k, new_col):
    # Attach hist[value_col] from k months before each row's month
    h = hist[["Zip Code", "Period", value_col]].copy()
    h["Period"] = h["Period"] + k            # move history forward k months to align
    return df.merge(h.rename(columns={value_col: new_col}), on=["Zip Code", "Period"], how="left")


for hist, col, short in [(cf_hist, "CalFresh_Dollars", "CalFresh"),
                         (fb_hist, "FoodBank_Pounds", "FoodBank")]:
    df = lagged_value(df, hist, col, 1, f"{short}_lag1")
    df = lagged_value(df, hist, col, 12, f"{short}_same_month_last_year")
    df = lagged_value(df, hist, col, 13, f"_{short}_lag13")
    df[f"{short}_yoy_change_lag1"] = df[f"{short}_lag1"] / df[f"_{short}_lag13"] - 1
    df = df.drop(columns=f"_{short}_lag13")

df = df.replace([np.inf, -np.inf], np.nan)

TEMPORAL_FEATURES = ["Month", "Quarter", "Month_sin", "Month_cos", "Months_Since_Start",
                     "CalFresh_lag1", "CalFresh_same_month_last_year", "CalFresh_yoy_change_lag1",
                     "FoodBank_lag1", "FoodBank_same_month_last_year", "FoodBank_yoy_change_lag1"]
df[TEMPORAL_FEATURES].describe().T

,count,mean,std,min,25%,50%,75%,max
Month,852.000,5.965,2.455,2.000,4.000,6.000,8.000,10.000
Quarter,852.000,2.320,0.896,1.000,2.000,2.000,3.000,4.000
Month_sin,852.000,0.011,0.768,-1.000,-0.866,0.000,0.866,1.000
Month_cos,852.000,-0.363,0.529,-1.000,-0.866,-0.500,0.000,0.500
Months_Since_Start,852.000,3.965,2.455,0.000,2.000,4.000,6.000,8.000
CalFresh_lag1,850.000,788972.521,889079.097,3470.000,162288.760,515164.395,975137.000,4724991.000
CalFresh_same_month_last_year,851.000,957581.584,1033253.744,3970.000,201691.500,622794.000,1231192.500,4742697.000
CalFresh_yoy_change_lag1,849.000,-0.155,0.234,-0.582,-0.328,-0.275,0.104,0.534
FoodBank_lag1,680.000,82934.752,111522.433,424.000,16207.000,47973.939,104818.905,776447.031
FoodBank_same_month_last_year,672.000,89821.509,123288.699,340.000,18374.868,48992.482,115219.066,909942.079


## 6. Policy change binary indicators (Assignment 4)

**Rationale:** 2023 included the end of CalFresh Emergency Allotments (April), ABAWD work rules resuming (July) and a LIHEAP FY24 funding cut (October). These can shift assistance levels independently of need. The indicators come straight from the Assignment 4 policy annotations in `TimeLag_PolicyAnnotated.csv`.

- `*_Policy_Active` columns are converted from Yes/No to 1/0.
- Columns that never vary in this window (every month is "active") carry no information and are dropped.

**Missing values:** none. Policy status is defined for every month.

In [7]:
POLICY_CANDIDATES = ["CalFresh_Policy_Active", "LIHEAP_Policy_Active", "FoodBank_Policy_Active",
                     "Any_Policy_Active", "CalFresh_Policy_Change", "LIHEAP_Policy_Change",
                     "FoodBank_Policy_Change", "Any_Policy_Change", "Policy_Period_EA_Ended",
                     "Policy_Period_ABWD_Resumed", "Policy_Period_LIHEAP_FY24_Cut",
                     "LIHEAP_Direction_Increased", "LIHEAP_Direction_Decreased"]

for c in POLICY_CANDIDATES:
    if not pd.api.types.is_numeric_dtype(df[c]):
        df[c] = df[c].map({"Yes": 1, "No": 0})
    df[c] = df[c].fillna(0).astype(int)

constant = [c for c in POLICY_CANDIDATES if df[c].nunique() == 1]
POLICY_FEATURES = [c for c in POLICY_CANDIDATES if c not in constant]
print("Dropped (no variation in this window):", constant)

df.groupby("Month_Year")[POLICY_FEATURES].first()

Dropped (no variation in this window): ['CalFresh_Policy_Active', 'LIHEAP_Policy_Active', 'Any_Policy_Active']


,FoodBank_Policy_Active,CalFresh_Policy_Change,LIHEAP_Policy_Change,FoodBank_Policy_Change,Any_Policy_Change,Policy_Period_EA_Ended,Policy_Period_ABWD_Resumed,Policy_Period_LIHEAP_FY24_Cut,LIHEAP_Direction_Increased,LIHEAP_Direction_Decreased
Month_Year,,,,,,,,,,
2023-02,1,0,1,0,1,0,0,0,1,0
2023-03,1,0,0,0,0,0,0,0,1,0
2023-04,1,1,0,1,1,1,0,0,1,0
2023-05,1,0,0,0,0,1,0,0,1,0
2023-06,1,0,0,0,0,1,0,0,1,0
2023-07,1,1,0,1,1,1,1,0,1,0
2023-08,1,0,0,0,0,1,1,0,1,0
2023-09,1,1,0,0,1,1,1,0,1,0
2023-10,0,1,1,1,1,1,1,1,0,1


## 7. Interaction terms

**Rationale:** earlier analysis showed the LIHEAP–food bank link is strong in high-poverty zip codes and absent in low-poverty ones, so the effect of LIHEAP depends on context. Interaction terms let a linear model capture this.

| Interaction | Why |
|---|---|
| `LIHEAP_lag1 × Poverty_Rate` | LIHEAP signal stronger where poverty is high |
| `LIHEAP_lag1 × Unemployment_Rate` | energy hardship plus job loss compounds food need |
| `LIHEAP_lag1 × Median_Rent_k` | high housing cost leaves less for food |
| `LIHEAP_lag1 × Policy_Period_EA_Ended` | loss of Emergency Allotments may change how LIHEAP need translates to food demand |
| `LIHEAP_lag1_per_1k_HH × Poverty_Rate` | size-adjusted version of the poverty interaction |

Continuous moderators are **centered** at their median before multiplying. This reduces collinearity with the main effects, and the LIHEAP main effect can then be read as its effect at a typical zip.

In [8]:
center = lambda s: s - s.median()

df["LIHEAPxPoverty"] = df["LIHEAP_lag1"] * center(df["Poverty_Rate"])
df["LIHEAPxUnemployment"] = df["LIHEAP_lag1"] * center(df["Unemployment_Rate"])
df["LIHEAPxRent"] = df["LIHEAP_lag1"] * center(df["Median_Rent_k"])
df["LIHEAPxEA_Ended"] = df["LIHEAP_lag1"] * df["Policy_Period_EA_Ended"]
df["LIHEAPperHHxPoverty"] = df["LIHEAP_lag1_per_1k_HH"] * center(df["Poverty_Rate"])

INTERACTION_FEATURES = ["LIHEAPxPoverty", "LIHEAPxUnemployment", "LIHEAPxRent",
                        "LIHEAPxEA_Ended", "LIHEAPperHHxPoverty"]
df[INTERACTION_FEATURES].describe().T

,count,mean,std,min,25%,50%,75%,max
LIHEAPxPoverty,687.000,19.534,94.529,-88.200,-5.900,0.000,11.250,915.000
LIHEAPxUnemployment,687.000,7.752,23.111,-31.900,-1.650,0.000,7.200,163.200
LIHEAPxRent,687.000,-0.330,4.449,-34.200,-1.065,0.000,0.933,20.628
LIHEAPxEA_Ended,687.000,5.435,9.009,0.000,0.000,1.000,8.000,75.000
LIHEAPperHHxPoverty,573.000,1.690,6.562,-6.563,-0.749,0.000,1.357,74.786


## 8. Consistent missing-value handling

One rule per type of missingness, applied the same way across all feature groups:

| Type | Example | Rule |
|---|---|---|
| Structural zero | LIHEAP `zip_gap` inside the data window | set to 0 (done in Section 3) |
| Outside data window | LIHEAP lags before Jan or after Jul 2023 | fill 0 **and** keep the `_observed` flag |
| No history | `*_lag1`, `*_same_month_last_year`, `*_yoy_change_lag1` | fill 0 **and** add a `_missing` flag |
| No census match | zips without ACS data | median across zips **and** `Demo_Imputed` flag (Section 4) |
| Derived from missing | interactions, per-household LIHEAP | fill 0 (the parent flags already mark these rows) |
| Target | `CalFresh_Dollars`, `FoodBank_Pounds` | **never imputed**; rows without a target are dropped at modeling time |

Filling with 0 *plus* a flag lets tree models and linear models both separate "zero" from "unknown". The table below shows missingness before and after.

In [9]:
FEATURES = LAG_FEATURES + DEMO_FEATURES + ["Demo_Imputed", "LIHEAP_lag1_per_1k_HH"] \
           + TEMPORAL_FEATURES + POLICY_FEATURES + INTERACTION_FEATURES

missing_before = df[FEATURES].isna().sum()

# Window-based LIHEAP gaps -> 0 (observed flags already exist)
for col in ["LIHEAP_t0"] + list(LAGS.values()):
    df[col] = df[col].fillna(0)
df["LIHEAP_lag_mean_3"] = df["LIHEAP_lag_mean_3"].fillna(0)
df["LIHEAP_lag_momentum"] = df["LIHEAP_lag_momentum"].fillna(0)

# History features -> 0 plus a missing flag
HISTORY = ["CalFresh_lag1", "CalFresh_same_month_last_year", "CalFresh_yoy_change_lag1",
           "FoodBank_lag1", "FoodBank_same_month_last_year", "FoodBank_yoy_change_lag1"]
for col in HISTORY:
    df[f"{col}_missing"] = df[col].isna().astype(int)
    df[col] = df[col].fillna(0)
FEATURES += [f"{c}_missing" for c in HISTORY]

# Derived features -> 0
for col in ["LIHEAP_lag1_per_1k_HH"] + INTERACTION_FEATURES:
    df[col] = df[col].fillna(0)

missing_after = df[FEATURES].isna().sum()
summary = pd.DataFrame({"missing_before": missing_before, "missing_after": missing_after}).fillna(0).astype(int)
assert missing_after.sum() == 0, "Unhandled missing values remain"
summary[summary["missing_before"] > 0]

,missing_before,missing_after
CalFresh_lag1,2,0
CalFresh_same_month_last_year,1,0
CalFresh_yoy_change_lag1,3,0
FoodBank_lag1,172,0
FoodBank_same_month_last_year,180,0
FoodBank_yoy_change_lag1,182,0
LIHEAP_lag1,165,0
LIHEAP_lag1_per_1k_HH,279,0
LIHEAP_lag2,154,0
LIHEAP_lag3,173,0


## 9. Feature dictionary

In [10]:
RATIONALE = {
    "LIHEAP_t0": ("LIHEAP lag", "Same-month enrollment; strongest within-zip link to CalFresh (r = 0.53)", "Outside window: 0 + flag"),
    "LIHEAP_lag1": ("LIHEAP lag", "Enrollment 1 month earlier; most useful leading indicator in the correlation analysis", "zip_gap: 0; outside window: 0 + flag"),
    "LIHEAP_lag2": ("LIHEAP lag", "Enrollment 2 months earlier; tests a longer lead time", "zip_gap: 0; outside window: 0 + flag"),
    "LIHEAP_lag3": ("LIHEAP lag", "Enrollment 3 months earlier; strongest cross-zip correlation with CalFresh (rho = 0.70)", "zip_gap: 0; outside window: 0 + flag"),
    "LIHEAP_lag_mean_3": ("LIHEAP lag", "Average of available lags; smooths noisy monthly counts", "Mean of available lags; 0 if none"),
    "LIHEAP_lag_momentum": ("LIHEAP lag", "Lag 1 minus lag 2; whether enrollment is rising or falling", "0 if either lag unknown"),
    "LIHEAP_t0_observed": ("LIHEAP lag", "1 if same-month LIHEAP is inside the data window", "None"),
    "LIHEAP_lag1_observed": ("LIHEAP lag", "1 if lag 1 is inside the data window", "None"),
    "LIHEAP_lag2_observed": ("LIHEAP lag", "1 if lag 2 is inside the data window", "None"),
    "LIHEAP_lag3_observed": ("LIHEAP lag", "1 if lag 3 is inside the data window", "None"),
    "Poverty_Rate": ("Demographic", "Core driver of need; moderates the LIHEAP-food bank link", "Median of zips + Demo_Imputed"),
    "Unemployment_Rate": ("Demographic", "Job loss raises demand for both programs", "Median of zips + Demo_Imputed"),
    "Median_Rent_k": ("Demographic", "Housing cost burden (thousands of $); weaker LIHEAP-food bank link in high-rent zips", "Median of zips + Demo_Imputed"),
    "Pct_Under_18": ("Demographic", "Households with children are prioritized by food programs", "Median of zips + Demo_Imputed"),
    "Pct_65_Plus": ("Demographic", "Older residents use programs differently (fixed incomes, lower CalFresh uptake)", "Median of zips + Demo_Imputed"),
    "Pct_Single_Parent_HH": ("Demographic", "Single-parent households face higher food insecurity", "Median of zips + Demo_Imputed"),
    "Pct_Living_Alone": ("Demographic", "Single-person households have different assistance patterns", "Median of zips + Demo_Imputed"),
    "Avg_HH_Size": ("Demographic", "Larger households need more food per enrollee", "Median of zips + Demo_Imputed"),
    "Log_Households": ("Demographic", "Controls for zip size, which inflates raw-total correlations", "Median of zips + Demo_Imputed"),
    "Demo_Imputed": ("Demographic", "1 if census values were median-imputed (no ACS match)", "None"),
    "LIHEAP_lag1_per_1k_HH": ("Demographic", "Size-adjusted LIHEAP; separates need from population", "0 if lag or households unknown"),
    "Month": ("Temporal", "Calendar month for seasonality", "None"),
    "Quarter": ("Temporal", "Quarter for broader seasonality", "None"),
    "Month_sin": ("Temporal", "Cyclical month encoding (December next to January)", "None"),
    "Month_cos": ("Temporal", "Cyclical month encoding (pair with Month_sin)", "None"),
    "Months_Since_Start": ("Temporal", "Linear trend across the study window", "None"),
    "CalFresh_lag1": ("Temporal", "Last month's CalFresh; strong persistence baseline", "0 + _missing flag"),
    "CalFresh_same_month_last_year": ("Temporal", "Seasonal baseline 12 months earlier", "0 + _missing flag"),
    "CalFresh_yoy_change_lag1": ("Temporal", "Year-over-year change using past values only (t-1 vs t-13)", "0 + _missing flag"),
    "FoodBank_lag1": ("Temporal", "Last month's food bank pounds; persistence baseline", "0 + _missing flag"),
    "FoodBank_same_month_last_year": ("Temporal", "Seasonal baseline 12 months earlier", "0 + _missing flag"),
    "FoodBank_yoy_change_lag1": ("Temporal", "Year-over-year change using past values only (t-1 vs t-13)", "0 + _missing flag"),
    "FoodBank_Policy_Active": ("Policy", "Any food bank policy in effect that month", "None"),
    "CalFresh_Policy_Change": ("Policy", "CalFresh policy changed that month", "None"),
    "LIHEAP_Policy_Change": ("Policy", "LIHEAP policy changed that month", "None"),
    "FoodBank_Policy_Change": ("Policy", "Food bank policy changed that month", "None"),
    "Any_Policy_Change": ("Policy", "Any program's policy changed that month", "None"),
    "Policy_Period_EA_Ended": ("Policy", "After CalFresh Emergency Allotments ended (Apr 2023)", "None"),
    "Policy_Period_ABWD_Resumed": ("Policy", "After ABAWD work rules resumed (Jul 2023)", "None"),
    "Policy_Period_LIHEAP_FY24_Cut": ("Policy", "After LIHEAP FY24 funding cut (Oct 2023)", "None"),
    "LIHEAP_Direction_Increased": ("Policy", "LIHEAP policy expected to raise enrollment", "None"),
    "LIHEAP_Direction_Decreased": ("Policy", "LIHEAP policy expected to lower enrollment", "None"),
    "LIHEAPxPoverty": ("Interaction", "LIHEAP effect stronger in high-poverty zips", "0 when lag 1 unknown"),
    "LIHEAPxUnemployment": ("Interaction", "Energy hardship plus job loss compounds food need", "0 when lag 1 unknown"),
    "LIHEAPxRent": ("Interaction", "High housing cost leaves less for food", "0 when lag 1 unknown"),
    "LIHEAPxEA_Ended": ("Interaction", "LIHEAP-food link may change after Emergency Allotments ended", "0 when lag 1 unknown"),
    "LIHEAPperHHxPoverty": ("Interaction", "Size-adjusted LIHEAP x poverty", "0 when inputs unknown"),
}
for col in HISTORY:
    RATIONALE[f"{col}_missing"] = ("Temporal", f"1 if {col} had no history and was filled with 0", "None")

feature_dict = pd.DataFrame(
    [(f, *RATIONALE.get(f, ("", "", ""))) for f in FEATURES],
    columns=["Feature", "Group", "Rationale", "Missing_value_rule"])
assert (feature_dict["Rationale"] != "").all(), "Every feature needs a documented rationale"

print(f"{len(FEATURES)} features:", feature_dict["Group"].value_counts().to_dict())
feature_dict

53 features: {'Temporal': 17, 'Demographic': 11, 'LIHEAP lag': 10, 'Policy': 10, 'Interaction': 5}


,Feature,Group,Rationale,Missing_value_rule
0,LIHEAP_t0,LIHEAP lag,Same-month enrollment; strongest within-zip li...,Outside window: 0 + flag
1,LIHEAP_lag1,LIHEAP lag,Enrollment 1 month earlier; most useful leadin...,zip_gap: 0; outside window: 0 + flag
2,LIHEAP_lag2,LIHEAP lag,Enrollment 2 months earlier; tests a longer le...,zip_gap: 0; outside window: 0 + flag
3,LIHEAP_lag3,LIHEAP lag,Enrollment 3 months earlier; strongest cross-z...,zip_gap: 0; outside window: 0 + flag
4,LIHEAP_lag_mean_3,LIHEAP lag,Average of available lags; smooths noisy month...,Mean of available lags; 0 if none
5,LIHEAP_lag_momentum,LIHEAP lag,Lag 1 minus lag 2; whether enrollment is risin...,0 if either lag unknown
6,LIHEAP_t0_observed,LIHEAP lag,1 if same-month LIHEAP is inside the data window,None
7,LIHEAP_lag1_observed,LIHEAP lag,1 if lag 1 is inside the data window,None
8,LIHEAP_lag2_observed,LIHEAP lag,1 if lag 2 is inside the data window,None
9,LIHEAP_lag3_observed,LIHEAP lag,1 if lag 3 is inside the data window,None


## 10. Save the feature set

In [11]:
out = df[KEY + TARGETS + FEATURES].sort_values(KEY).reset_index(drop=True)
out.to_csv("Predictive_Features.csv", index=False)
feature_dict.to_csv("Feature_Dictionary.csv", index=False)

print(f"Predictive_Features.csv: {out.shape[0]} rows x {out.shape[1]} columns")
print(f"Rows with a CalFresh target: {out['CalFresh_Dollars'].notna().sum()}; "
      f"with a food bank target: {out['FoodBank_Pounds'].notna().sum()}")
out.head()

Predictive_Features.csv: 852 rows x 57 columns
Rows with a CalFresh target: 850; with a food bank target: 681


,Zip Code,Month_Year,CalFresh_Dollars,FoodBank_Pounds,LIHEAP_t0,LIHEAP_lag1,LIHEAP_lag2,LIHEAP_lag3,LIHEAP_lag_mean_3,LIHEAP_lag_momentum,LIHEAP_t0_observed,LIHEAP_lag1_observed,LIHEAP_lag2_observed,LIHEAP_lag3_observed,Poverty_Rate,Unemployment_Rate,Median_Rent_k,Pct_Under_18,Pct_65_Plus,Pct_Single_Parent_HH,Pct_Living_Alone,Avg_HH_Size,Log_Households,Demo_Imputed,LIHEAP_lag1_per_1k_HH,Month,Quarter,Month_sin,Month_cos,Months_Since_Start,CalFresh_lag1,CalFresh_same_month_last_year,CalFresh_yoy_change_lag1,FoodBank_lag1,FoodBank_same_month_last_year,FoodBank_yoy_change_lag1,FoodBank_Policy_Active,CalFresh_Policy_Change,LIHEAP_Policy_Change,FoodBank_Policy_Change,Any_Policy_Change,Policy_Period_EA_Ended,Policy_Period_ABWD_Resumed,Policy_Period_LIHEAP_FY24_Cut,LIHEAP_Direction_Increased,LIHEAP_Direction_Decreased,LIHEAPxPoverty,LIHEAPxUnemployment,LIHEAPxRent,LIHEAPxEA_Ended,LIHEAPperHHxPoverty,CalFresh_lag1_missing,CalFresh_same_month_last_year_missing,CalFresh_yoy_change_lag1_missing,FoodBank_lag1_missing,FoodBank_same_month_last_year_missing,FoodBank_yoy_change_lag1_missing
0,91901,2023-03,355789.000,51489.000,2.000,1.000,0.000,0.000,0.500,1.000,1,1,1,0,10.200,5.400,2.064,20.984,17.863,5.543,18.384,2.866,8.762,0,0.157,3,1,1.000,0.000,1,347664.000,305561.000,0.135,23782.000,18282.000,-0.400,1,0,0,0,0,0,0,0,1,0,0.600,0.000,-0.079,0.000,0.094,0,0,0,0,0,0
1,91901,2023-04,230000.000,26145.000,1.000,2.000,1.000,0.000,1.000,1.000,1,1,1,1,10.200,5.400,2.064,20.984,17.863,5.543,18.384,2.866,8.762,0,0.313,4,2,0.866,-0.500,2,355789.000,314527.000,0.164,51489.000,37923.000,1.816,1,1,0,1,1,1,0,0,1,0,1.200,0.000,-0.158,2.000,0.188,0,0,0,0,0,0
2,91901,2023-05,230786.000,27788.000,2.000,1.000,2.000,1.000,1.333,-1.000,1,1,1,1,10.200,5.400,2.064,20.984,17.863,5.543,18.384,2.866,8.762,0,0.157,5,2,0.500,-0.866,3,230000.000,308579.000,-0.269,26145.000,24408.000,-0.311,1,0,0,0,0,1,0,0,1,0,0.600,0.000,-0.079,1.000,0.094,0,0,0,0,0,0
3,91901,2023-06,219267.000,17225.000,2.000,2.000,1.000,2.000,1.667,1.000,1,1,1,1,10.200,5.400,2.064,20.984,17.863,5.543,18.384,2.866,8.762,0,0.313,6,2,0.000,-1.000,4,230786.000,303402.000,-0.252,27788.000,47290.000,0.138,1,0,0,0,0,1,0,0,1,0,1.200,0.000,-0.158,2.000,0.188,0,0,0,0,0,0
4,91901,2023-07,207889.590,20383.000,1.000,2.000,2.000,1.000,1.667,0.000,1,1,1,1,10.200,5.400,2.064,20.984,17.863,5.543,18.384,2.866,8.762,0,0.313,7,3,-0.500,-0.866,5,219267.000,304693.000,-0.277,17225.000,15269.583,-0.636,1,1,0,1,1,1,1,0,1,0,1.200,0.000,-0.158,2.000,0.188,0,0,0,0,0,0


### Notes for modeling
- **Split by time, not at random** (for example train on Feb–Jul, test on Aug–Oct) so no future months leak into training.
- `LIHEAP_t0` is same-month data. Drop it for true forecasting, where next month's enrollment isn't known yet.
- Months after July 2023 have no observed LIHEAP lags (`*_observed = 0`). Consider training only on rows where `LIHEAP_lag1_observed == 1`, or compare models with and without these rows.
- Features overlap (lags with their mean; interactions with their main effects). Use regularization (ridge or lasso) or tree models, and check VIFs for any plain linear model.